# 👁️ AlgorimSeek-VL — Vision-Language Model Training
**Model:** Qwen2.5-VL / Qwen2-VL (7B / 3B / 2B)  
**Dataset:** `algorim_vision_training.json` (1 034 records) + `200_image_dataset` (200 code PNGs)  
**Method:** VLM QLoRA fine-tuning via Unsloth FastVisionModel  
**Output:** LoRA → Merged HF model → GGUF F16 → GGUF Q4_K_M


## ⚙️ 1 · GPU Check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 📦 2 · Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install datasets transformers sentencepiece Pillow torchvision huggingface_hub
print("✅ All packages installed")


## 📁 3 · Upload & Extract Datasets

In [ ]:
import os, zipfile, json, glob
from google.colab import files

# ── Upload ─────────────────────────────────────────────────────────
print("Upload: 200_image_dataset.zip + Datasets.zip")
uploaded = files.upload()

# ── Or mount Drive ─────────────────────────────────────────────────
# from google.colab import drive; drive.mount('/content/drive')
# DRIVE = '/content/drive/MyDrive/AlgorimDatasets'
# !cp {DRIVE}/200_image_dataset.zip /content/
# !cp {DRIVE}/Datasets.zip /content/

os.makedirs('/content/datasets', exist_ok=True)
os.makedirs('/content/images', exist_ok=True)

for z in ['200_image_dataset.zip', 'Datasets.zip']:
    if os.path.exists(f'/content/{z}'):
        with zipfile.ZipFile(f'/content/{z}') as zf:
            zf.extractall('/content/datasets')
        print(f"✅ Extracted {z}")

# Move all PNGs to /content/images
for png in glob.glob('/content/datasets/**/*.png', recursive=True):
    fname = os.path.basename(png)
    os.rename(png, f'/content/images/{fname}')

imgs   = glob.glob('/content/images/*.png')
jsons  = glob.glob('/content/datasets/**/*.json', recursive=True)
print(f"\n📸 Images: {len(imgs)}")
print(f"📂 JSON files: {[os.path.basename(j) for j in jsons]}")


## 🧩 4 · Build VL Dataset

In [ ]:
import json, glob, os
from PIL import Image
from datasets import Dataset

# ── Load vision_training.json ──────────────────────────────────────
vl_json = [f for f in glob.glob('/content/datasets/**/*.json', recursive=True)
           if 'vision' in f.lower()]
if not vl_json:
    vl_json = glob.glob('/content/datasets/**/*.json', recursive=True)
    vl_json = [f for f in vl_json if 'vision' in f]

records = []
if vl_json:
    with open(vl_json[0]) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except:
                    pass
print(f"✅ Loaded {len(records)} vision records")
print(f"   Keys: {list(records[0].keys())}")

# ── Map image_path → actual file ─────────────────────────────────
IMG_DIR = '/content/images'
available_imgs = {os.path.basename(f): f for f in glob.glob(f'{IMG_DIR}/*.png')}
print(f"   Available images: {len(available_imgs)}")

# ── Build samples ─────────────────────────────────────────────────
def extract_img_name(image_path):
    return os.path.basename(image_path)

samples = []
missing = 0
for rec in records:
    img_name = extract_img_name(rec.get('image_path', ''))
    if img_name not in available_imgs:
        missing += 1
        continue
    img_path = available_imgs[img_name]
    samples.append({
        "id":         rec['id'],
        "image_path": img_path,
        "prompt":     rec['prompt'],
        "response":   rec['response'],
        "category":   rec.get('category', ''),
        "difficulty": rec.get('difficulty', ''),
    })

print(f"   Matched: {len(samples)} / {len(records)} (missing images: {missing})")
print("\nSample:")
s = samples[0]
print(f"  image:    {s['image_path']}")
print(f"  prompt:   {s['prompt']}")
print(f"  response: {s['response'][:120]}...")


## 🤖 5 · Model Selection

In [ ]:
# ── Choose VL model ───────────────────────────────────────────────
VL_MODELS = {
    "Qwen2.5-VL-7B": "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit",
    "Qwen2.5-VL-3B": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "Qwen2-VL-7B":   "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "Qwen2-VL-2B":   "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
}

SELECTED    = "Qwen2.5-VL-7B"   # ← change this
BASE_MODEL  = VL_MODELS[SELECTED]
OUTPUT_NAME = "AlgorimSeek-VL"
MAX_SEQ_LEN = 2048
LORA_R      = 16
LORA_ALPHA  = 32

print(f"✅ Selected: {SELECTED}")
print(f"   HF model: {BASE_MODEL}")


## 🔧 6 · Load Vision Model + Apply LoRA

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,
    load_in_4bit  = True,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

model.print_trainable_parameters()
print(f"\n✅ VL Model loaded: {SELECTED}")


## 📋 7 · Prepare HF Dataset with Images

In [ ]:
from PIL import Image
from datasets import Dataset
import torch

SYSTEM_VL = (
    "You are AlgorimSeek-VL, an AI vision assistant specialized in reading and understanding "
    "Algorim programming language (algo a47) code images. "
    "You can analyze code screenshots, identify algorithms, explain logic, "
    "and generate textual representations of visual code."
)

def build_conversation(sample):
    """Build Qwen2-VL style conversation with image."""
    img = Image.open(sample['image_path']).convert('RGB')
    conversation = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_VL}]
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": sample['prompt']}
            ]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample['response']}]
        }
    ]
    return conversation

# Build the full dataset
def collate_fn(examples):
    conversations = [build_conversation(s) for s in examples]
    texts = [
        tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        for conv in conversations
    ]
    images_list = []
    for s in examples:
        img = Image.open(s['image_path']).convert('RGB')
        images_list.append([img])

    batch = tokenizer(
        text=texts,
        images=images_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    # Mask prompt tokens — only train on assistant response
    labels = batch['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    batch['labels'] = labels
    return batch

hf_dataset = Dataset.from_list(samples)
split      = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds   = split['train']
eval_ds    = split['test']
print(f"Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")


## 🏋️ 8 · Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps        = 5,
        num_train_epochs    = 3,
        learning_rate       = 2e-4,
        fp16  = not torch.cuda.is_bf16_supported(),
        bf16  =     torch.cuda.is_bf16_supported(),
        logging_steps       = 10,
        evaluation_strategy = "steps",
        eval_steps          = 50,
        save_strategy       = "steps",
        save_steps          = 100,
        output_dir          = f"/content/{OUTPUT_NAME}-checkpoints",
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "cosine",
        seed                = 42,
        remove_unused_columns = False,
        dataset_text_field  = "",
        dataset_kwargs      = {"skip_prepare_dataset": True},
        report_to           = "none",
    ),
)

print("🚀 Starting VL training...")
stats = trainer.train()
print(f"\n✅ Training complete!  loss={stats.metrics['train_loss']:.4f}")


## 💾 9 · Save LoRA Adapter

In [ ]:
lora_path = f"/content/{OUTPUT_NAME}-lora"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✅ LoRA saved → {lora_path}")


## 🔀 10 · Merge + Save Full Model

In [ ]:
merged_path = f"/content/{OUTPUT_NAME}"
print("⏳ Merging LoRA weights...")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
import os
total = sum(os.path.getsize(os.path.join(r,f)) for r,d,fs in os.walk(merged_path) for f in fs)
print(f"✅ Merged model → {merged_path}  ({total/1e9:.2f} GB)")


## 📤 11 · Export GGUF — F16

In [ ]:
gguf_path = f"/content/{OUTPUT_NAME}-GGUF"
print("⏳ Converting to GGUF F16...")
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="f16")
import glob, os
for f in glob.glob(f"{gguf_path}/*.gguf"):
    print(f"  {f}  ({os.path.getsize(f)/1e9:.2f} GB)")


## 📤 12 · Export GGUF — Q4_K_M

In [ ]:
print("⏳ Quantising to GGUF Q4_K_M...")
model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="q4_k_m")
import glob, os
for f in glob.glob(f"{gguf_path}/*.gguf"):
    print(f"  {f}  ({os.path.getsize(f)/1e9:.2f} GB)")
print("\n🎉 AlgorimSeek-VL pipeline complete!")


## 🧪 13 · Inference Test

In [ ]:
from PIL import Image
from unsloth import FastVisionModel

FastVisionModel.for_inference(model)

# Use first available test image
import glob
test_imgs = glob.glob('/content/images/*.png')
test_img  = Image.open(test_imgs[0]).convert('RGB') if test_imgs else None

if test_img:
    messages = [
        {"role":"system","content":[{"type":"text","text":"You are AlgorimSeek-VL, expert in Algorim code images."}]},
        {"role":"user","content":[
            {"type":"image","image":test_img},
            {"type":"text","text":"Read this Algorim code image and explain what it does:"}
        ]},
    ]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(text=text, images=[[test_img]], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print("=" * 60)
    print("AlgorimSeek-VL response:")
    print("=" * 60)
    print(response)
else:
    print("No test images found — upload images first.")
